# Tao: CUNP89T vs. Chang (ground truth)

Scores the **CUNP89T** Ruth/Jonah text against the **Chang** text, treating Chang as the reference. The verse alignment comes from `align_verses.py` (`benchmark_data/aligned-RUT-JON.json`), where a Chang verse range such as `1-2` is matched with the concatenation of the CUNP89T verses in that range.

Metrics follow `eval/flores/eval_nllb_gemini.py` and the other Tao notebook: corpus-level **chrF++** (`corpus_chrf(..., word_order=2)`) and **BLEU** (`corpus_bleu`), plus per-sentence chrF++.

The two texts are different translations of the same books, with different spelling (Naemey / Naomi, Poase / Poatse) and punctuation (Chang puts a space before punctuation and uses bracketed notes like `[10]`; CUNP89T uses curly quotes and `(10)`). So every score is reported twice: on the **raw** text and on a **normalized** text (lowercased, bracketed/parenthesised notes and punctuation removed). The normalized score shows how much of the gap is just formatting.

## 0. Setup

In [1]:
%pip install -q sacrebleu pandas


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from sacrebleu import corpus_chrf, corpus_bleu, sentence_chrf

pd.set_option("display.max_colwidth", 200)

DATA_FILE = Path("benchmark_data/aligned-RUT-JON.json")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

BOOK_NAMES = {"08": "Ruth", "32": "Jonah"}

## 1. Load the aligned data

Keys look like `BBCCCVVV-VVV` (book, chapter, first verse, last verse).

In [3]:
with open(DATA_FILE, encoding="utf-8") as f:
    aligned = json.load(f)

rows = []
for key, pair in aligned.items():
    m = re.fullmatch(r"(\d{2})(\d{3})(\d{3})-(\d{3})", key)
    bb, ch, v1, v2 = m.groups()
    rows.append({
        "key": key,
        "book": BOOK_NAMES.get(bb, bb),
        "chapter": int(ch),
        "verses": f"{int(v1)}" if v1 == v2 else f"{int(v1)}-{int(v2)}",
        "ref": pair["Chang"],      # ground truth
        "hyp": pair["CUNP89T"],    # system under test
    })
df = pd.DataFrame(rows)

print(f"{len(df)} aligned units;", df["book"].value_counts().to_dict())
empty = df[(df["ref"] == "") | (df["hyp"] == "")]
print(f"units with an empty side: {len(empty)}")
df.head()

127 aligned units; {'Ruth': 79, 'Jonah': 48}
units with an empty side: 0


,key,book,chapter,verses,ref,hyp
0,08001001-002,Ruth,1,1-2,"No kaka wa am amyan so amyokod no tao do iseli a tao no ka nag am mi kabcil amyan so asa ka tao do ivata , a mingaran so Ilimileh ,a mivahey do yota do poliheng ;mapalolo so mavakes na kano rarawa...",No kapeypanirsirngen da pa no mapacicici a tao do Yotay am amyan so rako a abcil do peycyeylilyan na. Am amyan so asa ka tao a mapalolo so mavakes na kano rarwa ka anak na mangay meyvahay do Podih...
1,08001003-003,Ruth,1,3,"Ma wakwak si Ilimileh o mavakes na rana kano rarwa ka anak na o ni mavezed ,",Mownay am mawakwak rana o kahanghangan ni Naomi a meyngaran so Idimele. So nikamimin da rana mavakes a kano rarwa ka anak na mahakay.
2,08001004-004,Ruth,1,4,"O rarwa ka anak na am ni mapasakey so mavakes do dang o ngaran da am si Earpa kani Loteh toda ji makasa ngaran[10] so awan na am ,",O rarwa ka anak na am mapazagpit sira so mavakes do Moab. O asa ka mahakay am meyngaran si Opeypa. O asa am meyngaran si Lote. So niyan da do dang so asa ngarnan (10) so awawan.
3,08001005-005,Ruth,1,5,Mawakwak si malon a kani Cilyan si naomi rana o ni mavezed a bo rana kahanghangan na abo anak .,Mawakwak rana sira Malon a kani Kedisay am to rana meyyonong o ikavyay no pahad da. So nikamimin da rana ni Naomi a abo so mahakay a kano anak.
4,08001006-006,Ruth,1,6,No keyyan na ni Naemey do moya am ni ma mizeng na rana o ka nimzat na ni Ama ta do to do monganey ja tao ori ikapya na no ni mahap da mowamowa da am ori avavatan na ni naemi a yangey o rarwa ka ic...,Meyratateng am mamizing na si Ama ta do to a marilaw no Isedi a tao a manta nira so kanen da. Am ori o nipeysinaoli da rana no rarwa ka anak da apo do Moab.


## 2. Normalization

Applied to both sides for the "normalized" scores: Unicode NFKC, lowercase, drop `[10]` / `(10)` / `(12,000,000)` style notes, drop all punctuation, collapse whitespace.

In [4]:
NOTE_RE = re.compile(r"[\[(][\d,\s]+[\])]")

def normalize(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).lower()
    text = NOTE_RE.sub(" ", text)
    text = "".join(" " if unicodedata.category(c).startswith("P") else c for c in text)
    return re.sub(r"\s+", " ", text).strip()

df["ref_norm"] = df["ref"].map(normalize)
df["hyp_norm"] = df["hyp"].map(normalize)
df[["ref", "ref_norm"]].iloc[[2, 40]]

,ref,ref_norm
2,"O rarwa ka anak na am ni mapasakey so mavakes do dang o ngaran da am si Earpa kani Loteh toda ji makasa ngaran[10] so awan na am ,",o rarwa ka anak na am ni mapasakey so mavakes do dang o ngaran da am si earpa kani loteh toda ji makasa ngaran so awan na am
40,"Manireng si Loteh am : "" na ni panci yaken na maci pangap do na pivazain a tao so mogi yato da ngey do naikavos .""",manireng si loteh am na ni panci yaken na maci pangap do na pivazain a tao so mogi yato da ngey do naikavos


## 3. Corpus-level scores

chrF++ with `word_order=2` and BLEU, exactly as in the other evaluation scripts.

In [5]:
def corpus_scores(frame, hyp_col, ref_col):
    hyps, refs = frame[hyp_col].tolist(), [frame[ref_col].tolist()]
    return {
        "chrF++": corpus_chrf(hyps, refs, word_order=2).score,
        "BLEU": corpus_bleu(hyps, refs).score,
    }

summary = pd.DataFrame({
    "raw": corpus_scores(df, "hyp", "ref"),
    "normalized": corpus_scores(df, "hyp_norm", "ref_norm"),
}).T.round(2)
summary["n"] = len(df)
summary

,chrF++,BLEU,n
raw,33.86,9.46,127
normalized,37.06,11.81,127


## 4. By book and chapter

In [6]:
def grouped(by):
    out = []
    for name, g in df.groupby(by, sort=False):
        raw = corpus_scores(g, "hyp", "ref")
        norm = corpus_scores(g, "hyp_norm", "ref_norm")
        out.append({
            "group": name if isinstance(name, str) else " ".join(map(str, name)),
            "n": len(g),
            "chrF++ raw": raw["chrF++"], "BLEU raw": raw["BLEU"],
            "chrF++ norm": norm["chrF++"], "BLEU norm": norm["BLEU"],
        })
    return pd.DataFrame(out).set_index("group").round(2)

display(grouped("book"))
display(grouped(["book", "chapter"]))

,n,chrF++ raw,BLEU raw,chrF++ norm,BLEU norm
group,,,,,
Ruth,79,32.60,7.74,35.75,10.14
Jonah,48,36.23,12.45,39.48,14.68


,n,chrF++ raw,BLEU raw,chrF++ norm,BLEU norm
group,,,,,
Ruth 1,21,35.65,9.85,39.22,12.90
Ruth 2,22,30.74,6.24,33.78,8.13
Ruth 3,18,31.27,7.51,33.52,9.02
Ruth 4,18,32.76,6.78,36.31,9.63
Jonah 1,17,35.83,11.64,38.96,13.79
Jonah 2,10,33.35,11.34,35.32,13.90
Jonah 3,10,38.03,13.95,42.05,16.82
Jonah 4,11,37.41,12.18,41.11,14.63


## 5. Per-sentence chrF++

Sentence-level chrF++ on the raw and normalized text. Low values mark verses where the two translations diverge the most.

In [7]:
df["chrf_raw"] = [sentence_chrf(h, [r], word_order=2).score for h, r in zip(df["hyp"], df["ref"])]
df["chrf_norm"] = [sentence_chrf(h, [r], word_order=2).score for h, r in zip(df["hyp_norm"], df["ref_norm"])]
df["len_ratio"] = (df["hyp_norm"].str.split().str.len() / df["ref_norm"].str.split().str.len().clip(lower=1)).round(2)

display(df[["chrf_raw", "chrf_norm", "len_ratio"]].describe().round(2))
print("Normalization gain (mean sentence chrF++):",
      round(df["chrf_norm"].mean() - df["chrf_raw"].mean(), 2))

,chrf_raw,chrf_norm,len_ratio
count,127.00,127.00,127.00
mean,33.76,36.93,1.04
std,7.10,7.80,0.34
min,20.08,21.21,0.42
25%,28.57,31.19,0.86
50%,34.28,36.32,1.00
75%,38.20,41.83,1.12
max,50.73,56.01,3.38


Normalization gain (mean sentence chrF++): 3.17


In [8]:
cols = ["key", "chrf_norm", "len_ratio", "ref", "hyp"]
print("Lowest-scoring verses")
display(df.nsmallest(5, "chrf_norm")[cols])
print("Highest-scoring verses")
display(df.nlargest(5, "chrf_norm")[cols])

Lowest-scoring verses


,key,chrf_norm,len_ratio,ref,hyp
44,08003002-002,21.210230,3.38,Mo pa manakem si Poase a zipos ta ?,"Yamo masangib a yamacivahavahay do lamlamsoy ni Poatse. Manngwa, beken a zipos na yaten ni Poatse? Cyaha, ta yamyan do rarahan sicyaraw eya a yameymowa so mogi."
27,08002007-007,21.883327,0.94,"Na nyahes yamen na amya sya amyan do panowjan na mi kala so ni makas a avyas no mogi .iya am yanim hep pa mai naka piwalam pa no nang ,a yami walam do sarowap . ""","Maniring o mavakes ori jya am, ‘To ko macihza do lamlamsoy ko a mangay omlangi so nipeysinsinmo da tao.’ So na ikanoyong a yanisomdep do vahay sicyaikwaya. So na ikabo no vazay na.”"
98,32002003-003,21.989366,0.53,Mo nyakas yaken do ka raratan ori rana am rakwa ka liliyan ; so na limotan jaken no rakwa wawa a yam hakehakeb na rana to hangopa yaken so katao tao.,"Mo nipangay yaken do gazawagaw no awa, kato mo na ngilikan jyaken no rako a ranom."
100,32002005-005,22.785121,1.00,Na yaken limoten no rakwa wawa ko ka jehmesan rana; na limoten yaken no kazngazngatan yama vawovawod o oo ko no tamek no wawa.,So nikatey yaro no ranom a komovot jyaken. So icyakma ney atbod a omlimot jyaken. So icyakma ney tamek a nimonong do oo ko.
91,32001013-013,23.290193,1.47,Am ji dam tadan o kakawod da sya do kacyan am jida kawod ta tey wara rana o abkas.,"Am ori o nikakzeban da no tao no avang a mapakarala so avang da do pamsan na am ji da pakongo, ta ikarahet na sira no paopaong na."


Highest-scoring verses


,key,chrf_norm,len_ratio,ref,hyp
79,32001001-001,56.008204,0.89,Meyded na o cireng ni Ama ta do to do anak ni yamitay a mingaran so yona manireng am :,"O ciring na ni Ama ta do to do anak ni Amitay a meyngaran so Yena am,"
104,32002009-009,53.037760,1.13,"Am yaken am isrey ko imo a patanangan ko so nakem ko o cireng ko jimo am pa ngononongan ko ta o igeygeglow am apo ji Ama ta do to.""",Am ori o sarayan ko nimo a patanangen ko so nakem ko jimo a patanangen jimo. O mina ciring ko jimo am pangononongan ko an. Ta si Ama ta do to o mapatazadang jyaten.”
95,32001017-017,52.399548,1.06,Ni manovoi si Ama ta do to so asa ka rakwa among a ni tomlen ni yona so niyan na do lotoi no among so makatlo arow [3] a kano makatlowa ahep.,Meyratateng am tovoin na ni Ama ta do to o asa ka rako a among a koman si Yena. So niyan na do velek no among so katlo araw a kano katlo ahep.
94,32001016-016,51.785844,1.28,Karahan na am ikanyaheyda rana no arwa tao si Ama ta do to a kapatanang da so nakem da si Ama ta do to a ka panci da no kararaley da nya.,Mamizing da no aro a tao o ciring na ni Ama ta do to am ikanyahey da rana si Ama ta do to. Am ori o nipatanang da so kagling a kano baka a patanangen da ji Ama ta do to.
106,32003001-001,51.345593,0.87,Ipirwa o cireng ni Ama ta do to a mangey ji yona am manireng am:,"Ipirwa o ciring na ni Ama ta do to ji Yena. Maniring am,"


## 6. Save results

In [9]:
df.drop(columns=["ref_norm", "hyp_norm"]).to_csv(RESULTS_DIR / "tao_cunp89t_vs_chang.csv", index=False)
summary.to_csv(RESULTS_DIR / "tao_cunp89t_vs_chang_summary.csv")
print("saved to", RESULTS_DIR.resolve())

saved to /home/paulhjwu/tulun/eval/tao/results
